> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 06 — The leakage audit, honest tuning and a model registry

**Sessions:** S11–S12 (Leakage & overfitting audit · Hyperparameters, ensembles & registry) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Run the audit: shuffled labels, a canary, a time shift and an overlap check.
2. Tune hyperparameters while logging every trial.
3. Average seeds instead of picking the lucky one.
4. Move a model through registry stages that refuse shortcuts.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()
from sklearn.metrics import roc_auc_score

## 1. The audit

Four tests, run on every model before anyone looks at its Sharpe ratio. First: **shuffle the labels** and refit under the same CV. A pipeline that still finds skill is leaking. Permute `y` with `np.random.default_rng(seed).permutation`, keep the index, and return `{"auc": mean CV AUC, "passed": |auc − 0.5| < tol}`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def shuffled_labels_test(model, X, y, cv, seed=0, tol=0.05):
    ys = pd.Series(np.random.default_rng(seed).permutation(y.to_numpy()), index=y.index)
    auc = float(p.cv_scores(model, X, ys, cv)["auc"].mean())
    return {"auc": auc, "passed": abs(auc - 0.5) < tol}

mk = p.regime_market()
ds = p.meta_dataset(mk)                                  # momentum meta-labels at CUSUM events (notebook 03)
Xm, ym = ds[p.FEATURES], ds["y"]
cv = p.PurgedKFold(ds["t1"], 5, 0.01)
rf = p.make_model("rf")
mine = [p.attempt(shuffled_labels_test, rf, Xm, ym, cv, s) for s in range(3)]
mine = p.check("shuffled_labels_test", mine, [p.shuffled_labels_test(rf, Xm, ym, cv, seed=s) for s in range(3)])
real = p.cv_scores(rf, Xm, ym, cv)["auc"].mean()
print(f"real labels: purged AUC {real:.3f};  3 shuffles: {[round(m['auc'], 3) for m in mine]}")

With about 800 events, one shuffle is noisy: some land below 0.45 and "fail" the fixed tolerance, and cross-validated AUCs under the null tend to sit slightly **below** 0.5 (each training fold's class mix mirrors its test fold's). Read it as a permutation test: the real model's AUC should sit above the spread of the shuffles, as it does here, if only just. Second: every training label must end before the test span starts or start after it ends. Count the training rows whose `[t0, t1]` overlaps `[first test t0, last test t1]`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def overlap_test(train_idx, test_idx, t1):
    t0 = t1.index
    lo, hi = t0[test_idx].min(), t1.iloc[test_idx].max()
    tr_t0, tr_t1 = t0[train_idx], t1.iloc[train_idx].to_numpy()
    n = int(((tr_t1 >= lo) & (tr_t0 <= hi)).sum())
    return {"n_overlap": n, "passed": n == 0}

from sklearn.model_selection import KFold

t1 = ds["t1"]
splits = {"purged fold 1": next(iter(cv.split(Xm))), "plain k-fold, fold 3": list(KFold(5).split(Xm))[2],
          "shuffled k-fold, fold 1": next(iter(p.ShuffledKFold(5).split(Xm)))}
mine = {k: p.attempt(overlap_test, tr, te, t1) for k, (tr, te) in splits.items()}
mine = p.check("overlap_test", mine, {k: p.overlap_test(tr, te, t1) for k, (tr, te) in splits.items()})
pd.DataFrame(mine).T

The last two tests are given. A **canary** (a feature that *is* the label plus noise) must come out as the most important feature, or the importance machinery is broken. A **time shift** (features one bar later) must not improve the AUC, or the features contain information from the future.

In [ ]:
print("canary:", p.canary_test(rf, Xm, ym, cv))
print("time shift:", {k: round(v, 3) if isinstance(v, float) else v for k, v in p.time_shift_test(rf, Xm, ym, cv).items()})

## 2. Tune, and count every trial

`p.random_search` tries 10 random settings of a gradient-boosting model (the labs use Optuna) and **logs every trial**: the number of trials goes into the Deflated Sharpe Ratio (Part 8).

In [ ]:
best, trials = p.random_search(Xm, ym, cv, n_trials=10)
print("best:", {k: round(v, 4) for k, v in best.items()}, f" (ln 2 = {np.log(2):.4f}: the log loss of always saying 50%)")
trials.round(4)

The winning trial's log loss is essentially ln 2, a coin flip: the search picked the model that learned the least (tiny learning rate, big leaves), because every model that learned more mostly learned noise. That is a result: there is little to find in these features, and 10 more trials now go into the research log.

## 3. Seeds are trials too

A random forest's AUC changes with its seed. Reporting the best seed is selection bias. Average the predicted probabilities of several seeds instead (`make(seed)` builds the model).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def seed_ensemble(make, X_train, y_train, X_test, seeds=(0, 1, 2, 3, 4)):
    return np.mean([make(s).fit(X_train, y_train).predict_proba(X_test)[:, 1] for s in seeds], axis=0)

n = len(ds)
cut = int(0.6 * n)
known = ds.iloc[:cut][ds["t1"].iloc[:cut] < ds.index[cut]]          # labels resolved before the test period
test = ds.iloc[cut:]
make = lambda s: p.make_model("rf", s)                                # noqa: E731
mine = p.check("seed_ensemble", p.attempt(seed_ensemble, make, known[p.FEATURES], known["y"], test[p.FEATURES]),
               p.seed_ensemble(make, known[p.FEATURES], known["y"], test[p.FEATURES]))
single = [roc_auc_score(test["y"], make(s).fit(known[p.FEATURES], known["y"]).predict_proba(test[p.FEATURES])[:, 1]) for s in range(5)]
print(f"test AUC of the 5 single seeds: {np.round(single, 3)};  the 5-seed ensemble: {roc_auc_score(test['y'], mine):.3f}")

## 4. A registry that says no

`p.ModelRegistry` is a small file-backed stand-in for MLflow. Versions start in `research` and move **one stage at a time** (`research → shadow → paper → live`), only if the audit passed. Promoting to live demotes the previous live version to paper.

In [ ]:
import tempfile
from pathlib import Path

reg = p.ModelRegistry(Path(tempfile.mkdtemp()) / "registry.json")
v1 = reg.register("meta_momentum", {"audit_passed": True, "cv_auc": round(float(real), 3), "trials": 10 + 5})
v2 = reg.register("meta_momentum", {"audit_passed": False, "cv_auc": 0.69, "note": "shuffled k-fold"})
for stage in ("shadow", "paper", "live"):
    reg.promote("meta_momentum", v1, stage)
for attempt, (version, stage) in {"skip a stage": (v2, "paper"), "failed audit": (v2, "shadow")}.items():
    try:
        reg.promote("meta_momentum", version, stage)
    except ValueError as err:
        print(f"{attempt}: refused ({err})")
print("live:", reg.get("meta_momentum", "live"))

## Wrap-up

* Audit before you believe: shuffled labels (as a permutation test), a canary, a time shift, no overlap.
* Log every trial and every seed; the best of many is biased upward.
* Promotion is a process with gates, not a decision in a notebook.
* Graded version: `labs/part10/week35_validation` (the audit, `optuna_search`, `seed_ensemble`, `ModelRegistry`) and Clinic W3 (the ML research report).